# Semana 07: Testes de Integração, Testes E2E e Quality Gates no GitHub Actions

## Módulo de Qualidade Contínua e Pipelines de CI/CD

Nesta aula, daremos o passo fundamental para transformar suítes de testes locais em uma esteira profissional de **Integração Contínua (CI)** automatizada na nuvem.

Na aula anterior (Semana 05), iniciamos o projeto `materiais/todo-app` implementando os **Testes Unitários** da aplicação. Hoje, avançaremos para as camadas superiores da **Pirâmide de Testes**:
1. **Testes de Integração:** Validação das rotas HTTP, controladores Flask, serialização JSON e códigos de status (`200`, `201`, `400`, `404`).
2. **Testes End-to-End (E2E):** Validação de jornadas completas de usuário simulando ciclos de vida reais na API To-Do.
3. **Quality Gates:** Estabelecimento de portões de qualidade automatizados com `pytest-cov` (barrando execuções com cobertura inferior a 80%).
4. **Novo Repositório no GitHub:** Publicação da aplicação To-Do em um novo repositório Git individual.
5. **Automação no GitHub Actions:** Criação do workflow `.github/workflows/ci.yml` para disparar automaticamente linting, testes e Quality Gates a cada `push` e `pull_request`.
6. **Simulação de Falha:** Vivência prática do bloqueio automático de Pull Requests quando um critério de qualidade não é atendido.

---


## 1. Fundamentação Teórica

### 1.1 A Pirâmide de Testes Aplicada a APIs REST

A Pirâmide de Testes (Mike Cohn / Martin Fowler) estabelece a proporção saudável de testes em um projeto de software. Abaixo, detalhamos como cada nível é mapeado no nosso projeto `todo-app`:

| Camada | Proporção | Escopo | Velocidade | O que valida no `todo-app` | Ferramentas |
| :--- | :---: | :--- | :---: | :--- | :--- |
| **Unitários** (Base) | ~70% | Funções e classes isoladas | Milissegundos | Regras de negócio da classe `Todo` e métodos do `TodoService` em memória. | `pytest` |
| **Integração** (Meio) | ~20% | Interação entre módulos | Segundos | Rotas HTTP `/api/todos`, parâmetros de URL, validação de payload 400 e status 201/404. | `pytest`, Flask `test_client` |
| **End-to-End / E2E** (Topo) | ~10% | Fluxo completo do cliente | Segundos/Minutos | Ciclo de vida completo: criar várias tarefas, listar, concluir uma, deletar outra e validar consistência final. | `pytest`, `test_client` |

---


### 1.2 Anatomia dos Testes de Integração com o Flask `test_client`

Diferente dos testes unitários, que chamam diretamente métodos Python como `service.create()`, os testes de integração simulam chamadas HTTP reais enviadas por clientes (Postman, navegadores ou aplicações front-end).

O Flask fornece um cliente de testes virtual (`test_client`) configurado no arquivo `tests/conftest.py`:

```python
@pytest.fixture
def app():
    app = create_app({"TESTING": True})
    global_todo_service.clear()   # Limpa o estado antes do teste
    yield app
    global_todo_service.clear()   # Limpa o estado apos o teste

@pytest.fixture
def client(app):
    return app.test_client()
```

#### Por que usar o `test_client`?
1. **Não abre portas de rede reais:** O `test_client` injeta a requisição diretamente no pipeline WSGI do Flask em memória, garantindo altíssima velocidade.
2. **Valida o contrato HTTP completo:** Testa se cabeçalhos (`Content-Type: application/json`), códigos de status HTTP (`200 OK`, `201 Created`, `400 Bad Request`, `404 Not Found`) e corpos de resposta JSON estão aderentes à especificação.
3. **Isolamento de Estado:** Através da cláusula `yield` nas fixtures, o repositório em memória é esvaziado antes e depois de cada teste, impedindo o acoplamento entre testes (*Test Pollution*).

---


### 1.3 Testes End-to-End (E2E) em APIs

Enquanto um teste de integração foca em um endpoint pontual (ex: *"testar se `POST /api/todos` retorna 201"*), os testes **End-to-End (E2E)** validam jornadas sequenciais do usuário final:

1. **Jornada de Ciclo de Vida Completo:**
   - Verifica se o sistema está saudável (`GET /health`).
   - Garante que a lista inicial é vazia.
   - Cadastra 3 tarefas com diferentes descrições.
   - Lista todas as tarefas e valida se todas foram persistidas.
   - Atualiza uma tarefa para concluída (`completed: true`).
   - Remove uma tarefa e confirma que uma busca subsequente por ela retorna `404 Not Found`.
   - Valida a consistência da lista final.

2. **Resiliência e Recuperação de Erros:**
   - Simula um cliente enviando requisições com falha intencional (payload vazio, título com espaços em branco).
   - Verifica se a API retorna o erro adequado (`400 Bad Request`).
   - Envia em seguida uma requisição válida e confirma que o serviço segue funcionando perfeitamente sem corrupção interna.

---


### 1.4 O Conceito e Mecânica de Quality Gates no CI

Um **Quality Gate** (Portão de Qualidade) é um conjunto de critérios objetivos e inegociáveis que o código deve satisfazer antes de ser mesclado nas branches protegidas (`develop` ou `main`).

```text
                       [Desenvolvedor abre Pull Request]
                                      │
                                      ▼
                        [Pipeline de CI no GitHub Actions]
                                      │
                    ┌─────────────────┴─────────────────┐
                    ▼                                   ▼
           [1. Analise Estatica]              [2. Execucao de Testes]
           - Flake8 (Sintaxe/PEP8)            - Testes Unitarios (100% OK)
           - Erros criticos == 0              - Testes Integracao (100% OK)
                                              - Testes E2E (100% OK)
                    │                                   │
                    └─────────────────┬─────────────────┘
                                      │
                                      ▼
                         [3. Portao de Cobertura]
                         pytest --cov-fail-under=80
                         Cobertura >= 80% do codigo
                                      │
                     ┌────────────────┴────────────────┐
                     ▼                                 ▼
               [APROVADO]                         [REPROVADO]
            Checks verdes no PR               Status Check com X vermelho
             Merge Permitido                   Merge Bloqueado no GitHub
```

#### O Parâmetro `--cov-fail-under=80`
O `pytest-cov` possui um parâmetro embutido para Quality Gate:
```bash
pytest --cov=app --cov-report=term-missing --cov-fail-under=80 tests/
```
- Se a cobertura total for **80% ou mais**, o Pytest finaliza com código de saída `0` (Sucesso).
- Se a cobertura for **menor que 80%** (ex: 79.9%), o Pytest força o código de saída `1` (Falha), interrompendo o pipeline do GitHub Actions imediatamente.

---


### 1.5 Arquitetura do GitHub Actions

O **GitHub Actions** é a plataforma de automação e CI/CD nativa do GitHub. Seus conceitos fundamentais incluem:

- **Workflow:** Procedimento automatizado definido em YAML dentro de `.github/workflows/`.
- **Events / Triggers (`on`):** Acontecimentos que disparam a execução (ex: `push`, `pull_request`).
- **Runner:** Máquina virtual hospedada pela nuvem do GitHub (ex: `ubuntu-latest`) que executa os jobs.
- **Jobs:** Conjunto de etapas (`steps`) executadas sequencialmente no mesmo runner.
- **Steps:** Comandos de terminal (`run`) ou ações pré-fabricadas reutilizáveis (`uses`).

---


## 2. Roteiro Prático Hands-On Passo a Passo

A atividade prática de hoje será realizada no projeto **`materiais/todo-app`**.

### Passo 1: Revisão da Estrutura e Execução dos Testes Unitários

Abra o terminal na pasta do projeto:
```bash
cd materiais/todo-app
```

Execute a suíte de testes unitários:
```bash
pytest -v tests/unit
```
> Observe a execução instantânea (milissegundos), validando a criação de entidades e o serviço em memória.

---


### Passo 2: Executar a Suíte de Testes de Integração

Execute os testes de integração das rotas HTTP:
```bash
pytest -v tests/integration
```

Os testes de integração validam:
1. `test_api_health_check`: Retorno `HTTP 200` e `{"status": "ok"}`.
2. `test_api_listar_tarefas_inicialmente_vazia`: Retorno `HTTP 200` e `[]`.
3. `test_api_criar_tarefa_com_sucesso`: Retorno `HTTP 201` com `id`, `title` e `completed: false`.
4. `test_api_criar_tarefa_sem_titulo_deve_retornar_400`: Retorno `HTTP 400` e chave `error`.
5. `test_api_obter_tarefa_por_id_existente`: Retorno `HTTP 200` com dados corretos.
6. `test_api_obter_tarefa_inexistente_deve_retornar_404`: Retorno `HTTP 404` para IDs não cadastrados.
7. `test_api_atualizar_tarefa_com_sucesso`: Retorno `HTTP 200` com status alterado.
8. `test_api_deletar_tarefa_com_sucesso`: Retorno `HTTP 200` e posterior `HTTP 404` no mesmo ID.

---


### Passo 3: Executar a Suíte de Testes End-to-End (E2E)

Execute os testes de jornada de ciclo de vida e resiliência:
```bash
pytest -v tests/e2e
```

---


### Passo 4: Executar Toda a Suíte e Validar o Quality Gate Localmente

Execute o comando com medição de cobertura e trava de Quality Gate:
```bash
pytest --cov=app --cov-report=term-missing --cov-fail-under=80 tests/
```

Exemplo de saída esperada:
```text
Name              Stmts   Miss  Cover   Missing
-----------------------------------------------
app/__init__.py      11      0   100%
app/models.py        10      0   100%
app/routes.py        45      2    96%   51-52
app/services.py      36      1    97%   28
-----------------------------------------------
TOTAL               102      3    97%
Required test coverage of 80% reached. Total coverage: 97.06%
============================= 20 passed in 0.35s ==============================
```

---


### Passo 5: Subir o To-Do App em um Novo Repositório no GitHub

Para que o GitHub Actions execute a esteira na nuvem, cada aluno deve subir a pasta `todo-app` em um repositório próprio no GitHub.

#### 1. No terminal da sua máquina (dentro de `todo-app`):
> **Atenção:** Certifique-se de que a pasta `todo-app` possui seu próprio versionamento independente.

```bash
# Inicializar o repositorio Git local
git init
git branch -M main

# Adicionar os arquivos e fazer o primeiro commit
git add .
git commit -m "feat: estrutura inicial da To-Do API com testes e CI"
```

#### 2. Criar o repositório no GitHub:
1. Acesse: **https://github.com/new**
2. Defina o nome do repositório: `todo-app-devops` (ou `todo-app-ci`).
3. Marque como **Público**.
4. **NÃO** marque as caixas "Add a README file", ".gitignore" ou "license" (o projeto já contém esses arquivos!).
5. Clique em **Create repository**.

#### 3. Conectar e fazer o push para o GitHub:
```bash
# Vincular o remoto (substitua SEU_USUARIO pelo seu username do GitHub)
git remote add origin https://github.com/SEU_USUARIO/todo-app-devops.git

# Enviar a branch main
git push -u origin main

# Criar e enviar a branch develop (seguindo o GitFlow)
git checkout -b develop
git push -u origin develop
```

---


### Passo 6: Criação do Workflow de CI no GitHub Actions

Agora que a aplicação e os testes estão prontos e versionados, vamos configurar a esteira de **Integração Contínua** criando o arquivo de workflow na raiz do seu projeto `todo-app`.

#### 1. Criar o diretório de workflows no terminal:
```bash
# No Windows PowerShell:
New-Item -ItemType Directory -Force .github/workflows

# No Linux, macOS ou Git Bash:
mkdir -p .github/workflows
```

#### 2. Criar o arquivo `.github/workflows/ci.yml`:
Crie o arquivo `.github/workflows/ci.yml` no seu editor (VS Code) e adicione o seguinte conteúdo YAML:

```yaml
name: CI Pipeline - To-Do API

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main, develop]

jobs:
  testes-e-quality-gate:
    name: Testes Automatizados e Quality Gate
    runs-on: ubuntu-latest

    steps:
      - name: 1. Checkout do Repositorio
        uses: actions/checkout@v4

      - name: 2. Configurar Ambiente Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"

      - name: 3. Instalar Dependencias
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: 4. Analise Estatica (Flake8)
        run: |
          flake8 app tests --count --select=E9,F63,F7,F82 --show-source --statistics

      - name: 5. Executar Testes Unitarios
        run: pytest -v tests/unit

      - name: 6. Executar Testes de Integracao
        run: pytest -v tests/integration

      - name: 7. Executar Testes E2E
        run: pytest -v tests/e2e

      - name: 8. Validar Quality Gate de Cobertura
        run: pytest --cov=app --cov-report=term-missing --cov-fail-under=80 tests/
```

#### 3. Fazer o commit e push do workflow para o GitHub:
```bash
git add .github/workflows/ci.yml
git commit -m "ci: adicionar workflow do GitHub Actions com testes e quality gate"
git push origin main
```

#### 4. Acompanhar a execução no GitHub:
1. Abra seu repositório no navegador.
2. Clique na aba **Actions** no topo.
3. Observe o workflow executando com ícone amarelo (em progresso) e em seguida verde (sucesso).
4. Clique no job para expandir e inspecionar os logs de cada step!

---


### Passo 7: Simulação de Falha do Quality Gate em Pull Request

Para constatar o valor de proteção do Quality Gate:

1. Crie uma branch de feature no terminal:
   ```bash
   git checkout -b feature/teste-quebra-gate
   ```
2. Abra o arquivo `app/routes.py` e altere uma linha propositalmente (exemplo: mude a rota `/health` para retornar status `500` em vez de `200`).
3. Faça commit e push:
   ```bash
   git commit -am "fix: alteracao que quebra o contrato da API"
   git push -u origin feature/teste-quebra-gate
   ```
4. Acesse o GitHub e abra um **Pull Request** da branch `feature/teste-quebra-gate` para a branch `develop`.
5. Observe o GitHub Actions disparar imediatamente.
6. Em poucos segundos, o Step de Testes de Integração falhará com **X vermelho** e o GitHub mostrará:
   > **"All checks have failed - Merging is blocked"**

Isso demonstra como o pipeline impede que código com defeito chegue à branch principal!

---


## 3. Explicação Detalhada da Engenharia da Pipeline (`.github/workflows/ci.yml`)

Nesta seção, dissecamos a engenharia por trás de cada linha do arquivo de workflow. Compreender a finalidade de cada parâmetro é fundamental para projetar esteiras seguras, performáticas e resilientes em ambientes profissionais de DevOps.

### 3.1 Diagrama de Fluxo e Execução da Pipeline

```text
       [Disparo: Push ou Pull Request na main / develop]
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │         PROVISIONAMENTO DO RUNNER VIRTUAL              │
  │               (GitHub Cloud: ubuntu-latest)            │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 1: actions/checkout@v4                            │
  │ -> Clona o repositório no commit exato                 │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 2: actions/setup-python@v5                        │
  │ -> Provisiona Python 3.11 e restaura cache do pip      │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 3: Instalação de Dependências                     │
  │ -> Atualiza pip e instala requirements.txt             │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 4: Shift-Left Linting com Flake8                  │
  │ -> Falha rápida se houver erros críticos de sintaxe    │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 5: Testes Unitários (pytest -v tests/unit)        │
  │ -> Validação atômica de models e services em memória   │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 6: Testes de Integração (pytest tests/integration)│
  │ -> Validação de rotas HTTP com Flask test_client       │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 7: Testes E2E (pytest -v tests/e2e)               │
  │ -> Validação da jornada completa de ponta a ponta      │
  └───────────────────────────┬────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────┐
  │ Step 8: Quality Gate de Cobertura de Código            │
  │ pytest --cov=app --cov-fail-under=80                   │
  └───────────────────────────┬────────────────────────────┘
                              │
               ┌──────────────┴──────────────┐
               ▼                             ▼
        [Cobertura >= 80%]            [Cobertura < 80%]
          (Código 0)                    (Código 1)
               │                             │
               ▼                             ▼
       STATUS: SUCESSO               STATUS: FALHA
      Checks verdes no PR          Build quebra / PR travado
```

---

### 3.2 Análise Detalhada Bloco a Bloco do Workflow YAML

#### 1. Identificação e Gatilhos de Disparo (`name` e `on`)
```yaml
name: CI Pipeline - To-Do API

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main, develop]
```
- **`name`:** Define o nome do workflow exibido na interface visual do GitHub e nas badges de status do repositório.
- **Gatilho `push`:** Executa a esteira sempre que novos commits são enviados diretamente para `main` ou `develop`. Garante que o estado das branches principais permaneça íntegro após a mesclagem.
- **Gatilho `pull_request`:** É o **coração preventivo do CI**. Dispara a esteira **antes** do merge, montando uma branch virtual temporária (`refs/pull/:id/merge`). Isso testa como o código se comportará se o merge for aprovado, protegendo a equipe contra quebras surpresas.
- **Filtro de Branches (`[main, develop]`):** Alinhado ao **GitFlow**. As branches de funcionalidade (`feature/*`) são testadas automaticamente no momento em que abrem um PR contra `develop`.

---

#### 2. Ambiente de Execução e Runners (`jobs` e `runs-on`)
```yaml
jobs:
  testes-e-quality-gate:
    name: Testes Automatizados e Quality Gate
    runs-on: ubuntu-latest
```
- **`jobs`:** Unidade organizadora do GitHub Actions. Cada job roda em uma máquina virtual independente e pode rodar em paralelo com outros jobs.
- **`runs-on: ubuntu-latest`:** Instrução que requisita uma máquina virtual Linux Ubuntu (Runner gerenciado pelo GitHub). Esse ambiente é:
  * **Efêmero:** Criado do zero no início do job e completamente destruído ao final.
  * **Estéril (Clean Slate):** Não carrega arquivos, bancos de dados residuais ou pacotes de execuções anteriores, eliminando o clássico problema *"na minha máquina funciona"*.

---

#### 3. Step 1: Clonagem do Repositório (`actions/checkout@v4`)
```yaml
- name: 1. Checkout do Repositorio
  uses: actions/checkout@v4
```
- **Por que é estritamente necessário?** O runner provisionado pelo GitHub inicia com o sistema operacional limpo e disco vazio. Sem esse step, os arquivos do repositório (`app/`, `tests/`, `requirements.txt`) simplesmente não existem dentro da VM.
- **O que faz?** Executa um `git clone` otimizado apontando para o commit exato do disparo.

---

#### 4. Step 2: Runtime Python e Cache de Pacotes (`actions/setup-python@v5`)
```yaml
- name: 2. Configurar Ambiente Python 3.11
  uses: actions/setup-python@v5
  with:
    python-version: "3.11"
    cache: "pip"
```
- **`python-version: "3.11"`:** Padroniza a versão exata do interpretador, evitando discrepâncias causadas por funcionalidades novas ou depreciadas entre versões.
- **`cache: "pip"` (Otimização de Performance):**
  * O GitHub Actions gera uma chave hash baseada no conteúdo de `requirements.txt`.
  * Se as dependências não mudaram em relação ao último build, os pacotes são restaurados instantaneamente do cache local do GitHub, poupando minutos de download do PyPI a cada commit.

---

#### 5. Step 3: Instalação das Dependências
```yaml
- name: 3. Instalar Dependencias
  run: |
    python -m pip install --upgrade pip
    pip install -r requirements.txt
```
- **`python -m pip install --upgrade pip`:** Garante que o gerenciador de pacotes esteja na versão mais moderna, prevenindo falhas de compilação de binários (`wheels`).
- **`pip install -r requirements.txt`:** Instala de forma determinística `Flask`, `pytest`, `pytest-cov` e `flake8`.

---

#### 6. Step 4: Análise Estática de Código com Flake8 (*Fail-Fast*)
```yaml
- name: 4. Analise Estatica (Flake8)
  run: |
    flake8 app tests --count --select=E9,F63,F7,F82 --show-source --statistics
```
- **Princípio *Shift-Left Testing*:** Problemas devem ser detectados no estágio mais inicial possível.
- **Princípio *Fail-Fast*:** A checagem com linter dura centésimos de segundo. Se houver erro de sintaxe (`E9`), comparação incorreta (`F63`), bloco inválido (`F7`) ou variável indefinida (`F82`), o pipeline é abortado na hora, sem desperdiçar recursos executando suítes de testes complexas.

---

#### 7. Steps 5, 6 e 7: Execução Escalonada das Três Camadas de Testes
```yaml
- name: 5. Executar Testes Unitarios
  run: pytest -v tests/unit

- name: 6. Executar Testes de Integracao
  run: pytest -v tests/integration

- name: 7. Executar Testes E2E
  run: pytest -v tests/e2e
```
- **Por que separar em três steps em vez de um único comando?**
  1. **Isolamento e Rastreabilidade Visual:** Se um teste falhar, o GitHub Actions marca com X vermelho exatamente o step responsável, permitindo ao desenvolvedor saber instantaneamente se a falha é na regra de negócio (unitário), no contrato HTTP (integração) ou no fluxo sequencial (E2E).
  2. **Ordem de Complexidade e Custo:** Começa pelos testes que executam em milissegundos e avança progressivamente para as suítes mais abrangentes.

---

#### 8. Step 8: O Quality Gate com Medição de Cobertura
```yaml
- name: 8. Validar Quality Gate de Cobertura
  run: pytest --cov=app --cov-report=term-missing --cov-fail-under=80 tests/
```
- **`--cov=app`:** Mede a cobertura sobre os arquivos de código-fonte de produção em `app/`.
- **`--cov-report=term-missing`:** Gera a tabela detalhada no log indicando quais números de linha específicos não foram atingidos pelos testes.
- **`--cov-fail-under=80` (A Trava do Portão):**
  * Se a cobertura for **>= 80%**, o comando retorna status `0` (Success) e o job conclui com sucesso.
  * Se a cobertura for **< 80%** (ex: 79.5%), o comando retorna código de saída `1` (Failure), o step é marcado como reprovado e a esteira é interrompida.

---

### 3.3 Tabela Comparativa dos Steps da Esteira

| Step | Ferramenta | Objetivo Técnico | Critério de Aprovação |
| :--- | :--- | :--- | :--- |
| **1. Checkout** | `actions/checkout@v4` | Clonar o código-fonte no workspace | Clone sem erros |
| **2. Setup Python** | `actions/setup-python@v5` | Configurar o interpretador Python 3.11 | Ambiente pronto e cache configurado |
| **3. Dependências** | `pip install` | Instalar bibliotecas de produção e teste | Instalação concluída com saída 0 |
| **4. Linting** | `flake8` | Checar erros graves de sintaxe e código morto | 0 erros críticos encontrados |
| **5. Testes Unitários** | `pytest tests/unit` | Validar classes `Todo` e `TodoService` | 100% dos testes unitários passando |
| **6. Testes Integração** | `pytest tests/integration` | Validar rotas HTTP e contratos JSON | 100% das rotas respondendo corretamente |
| **7. Testes E2E** | `pytest tests/e2e` | Validar jornadas completas e resiliência | 100% dos fluxos ponta a ponta aprovados |
| **8. Quality Gate** | `pytest-cov` | Assegurar governança de cobertura | Cobertura total de código >= 80% |

---

### 3.4 Conexão com as Branch Protection Rules no GitHub

Para transformar essa automação em uma barreira intransponível contra defeitos em produção, ativamos a **Proteção de Branches** no repositório:

1. No repositório GitHub, acesse: **Settings > Branches > Add branch protection rule**.
2. Em **Branch name pattern**, defina `develop` (e depois `main`).
3. Marque a caixa **"Require status checks to pass before merging"**.
4. Na barra de busca que se abre, selecione o job: `Testes Automatizados e Quality Gate`.
5. Clique em **Save changes**.

> **Efeito Prático:** A partir desse momento, o botão de merge do Pull Request fica **desabilitado** até que o GitHub Actions execute todo o pipeline e confirme que todos os 8 steps passaram com sucesso!

---


## 4. Demonstração Interativa: Simulador do Pipeline e Quality Gate

A célula de código abaixo simula a execução sequencial das três camadas de testes e do validador de Quality Gate diretamente no notebook.


In [ ]:
# SIMULADOR DO PIPELINE DE QUALIDADE CONTINUA

def simular_execucao_pipeline(falha_proposital=False, cobertura_simulada=97.0, threshold_gate=80.0):
    steps = [
        ("Step 1: Flake8 Linter", True, "0 erros de sintaxe e variaveis indefinidas"),
        ("Step 2: Testes Unitarios (models + services)", True, "10 testes unitarios executados com sucesso"),
        ("Step 3: Testes de Integracao (rotas HTTP)", not falha_proposital, "Falha: esperado 200, recebido 500 em /health" if falha_proposital else "8 testes de integracao aprovados"),
        ("Step 4: Testes E2E (ciclo de vida CRUD)", not falha_proposital, "Abortado devido a falha previa" if falha_proposital else "2 fluxos completos aprovados")
    ]
    
    print("=" * 70)
    print("       RELATORIO DE EXECUCAO - PIPELINE GITHUB ACTIONS CI")
    print("=" * 70)
    
    todos_steps_ok = True
    for nome, status, detalhe in steps:
        tag = "[OK] PASSED " if status else "[X] FAILED  "
        print(f"{tag} | {nome:<42} | {detalhe}")
        if not status:
            todos_steps_ok = False
            break
            
    print("-" * 70)
    print(f"Meta Minima do Quality Gate: {threshold_gate:.1f}%")
    print(f"Cobertura de Codigo Obtida:  {cobertura_simulada:.1f}%")
    
    gate_aprovado = todos_steps_ok and (cobertura_simulada >= threshold_gate)
    print("-" * 70)
    if gate_aprovado:
        print(">>> STATUS FINAL DO QUALITY GATE: APROVADO")
        print(">>> STATUS DO PULL REQUEST: MERGE PERMITIDO NA BRANCH DEVELOP")
    else:
        print(">>> STATUS FINAL DO QUALITY GATE: REPROVADO")
        print(">>> STATUS DO PULL REQUEST: MERGE BLOQUEADO PELO GITHUB ACTIONS")
    print("=" * 70)


# 1. Execucao com Sucesso (Cenario Padrao do todo-app)
print("--- CENARIO 1: CODIGO CONFORME ---")
simular_execucao_pipeline(falha_proposital=False, cobertura_simulada=97.0, threshold_gate=80.0)

print("\n--- CENARIO 2: FALHA INTENCIONAL NO TESTE DE INTEGRACAO ---")
simular_execucao_pipeline(falha_proposital=True, cobertura_simulada=97.0, threshold_gate=80.0)


---

## 5. Exercícios de Fixação e Avaliação (Preparatórios para P1)

### Questão 1 (Pirâmide de Testes e APIs)
Explique a diferença fundamental entre um **Teste Unitário**, um **Teste de Integração** e um **Teste End-to-End (E2E)** no contexto de uma API REST como a nossa To-Do API. Em qual das camadas você testaria a validação de parâmetros de requisição HTTP (como `Content-Type` e status `400 Bad Request`)?

### Questão 2 (Isolamento e Fixtures no Pytest)
No arquivo `tests/conftest.py`, a fixture `app` utiliza a instrução `yield` em conjunto com `global_todo_service.clear()`. Qual é o risco para a confiabilidade dos testes de integração caso essa limpeza de estado não seja realizada antes e depois de cada teste?

### Questão 3 (Mecânica do Quality Gate)
Qual é a diferença entre executar `pytest --cov=app` e `pytest --cov=app --cov-fail-under=80` em um pipeline de CI? O que acontece com o status de execução de um job no GitHub Actions quando o comando retorna código de saída `1`?

### Questão 4 (GitHub Actions e GitFlow)
Analise o bloco de triggers abaixo de um arquivo de workflow:
```yaml
on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main, develop]
```
Por que é fundamental escutar o evento `pull_request` nas branches `develop` e `main` em vez de rodar o pipeline apenas após o `push` direto nessas branches?
